In [1]:
OPENAI_KEY = "INSERT_KEY"
GEMINI_KEY = "INSERT_KEY"
CLAUDE_KEY = "INSERT_KEY"

In [13]:
import pandas as pd

# Use this to show every single row
pd.set_option('display.max_rows', None)

# Use this to show every single column
pd.set_option('display.max_columns', None)

# Use this to show the full text content within a cell (no "...")
pd.set_option('display.max_colwidth', None)

In [14]:
import os, re, time
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

# --- 2. UTILITY FUNCTIONS ---
def super_clean_id(filename):
    nums = re.findall(r"(\d+)", str(filename))
    return str(int(nums[-1])) if nums else str(filename).split('.')[0].strip().lower()

def load_dir_robust(folder):
    texts = {}
    if not folder.is_dir(): return texts
    for fn in os.listdir(folder):
        if fn.lower().endswith(".txt"):
            eid = super_clean_id(fn)
            with open(folder / fn, "r", encoding="utf-8", errors="ignore") as f:
                texts[eid] = f.read().strip()
    return texts

def get_embeddings_batched(text_list, model_id):
    if "models/" in model_id:
        result = genai.embed_content(model=model_id, content=text_list, task_type="clustering")
        return np.array(result['embedding'])
    return SentenceTransformer(model_id).encode(text_list, show_progress_bar=False)


# --- 1. CONFIGURATION ---
genai.configure(api_key=GEMINI_KEY)

ARGREWRITE_ROOT = Path("../ArgRewrite").resolve()
D1_PATH = ARGREWRITE_ROOT / "dataset" / "essays" / "Draft1"
D2_PATH = ARGREWRITE_ROOT / "dataset" / "essays" / "Draft2"

EMBEDDING_MODELS = {
    "MiniLM-L6": "all-miniLM-L6-v2",
    "Gemini-004": "models/text-embedding-004"
}
# CUSTOMIZE LLM NAMES HERE
# Format: "FOLDER_NAME_UPPER": "Display Name"

# --- 1. CONFIGURATION ---
LLM_DISPLAY_NAMES = {
    "CLAUDE-HAIKU-4-5-20251001": "claude 3.5 haiku",
    "GEMINI-2.5-FLASH": "gemini 2.5 flash",
    "GPT-5-MINI": "gpt-5-mini",
    # "GPT-4O-MINI": "gpt-4o-mini"
}

VARIANT_COLORS = {
    "minimal": "#d95f02", "grammar": "#1b9e77", "general": "#b2182b",
    "completion": "#ce1256", "expansion": "#7570b3"
}

DESCRIPTIVE_NAMES = {
    "minimal": "Minimal Edit", "grammar": "Grammar Edits",
    "completion": "Completion", "expansion": "Expansion", "general": "General Revision",
}

PLOT_ORDER = ["general", "minimal", "grammar", "completion", "expansion"]

# --- 3. DATA AGGREGATION ---
d1_map, d2_map = load_dir_robust(D1_PATH), load_dir_robust(D2_PATH)
common_ids = sorted(set(d1_map.keys()) & set(d2_map.keys()), key=lambda x: int(x))
df_human = pd.DataFrame([{"essay_id": eid, "draft1": d1_map[eid], "draft2": d2_map[eid]} for eid in common_ids])

MODEL_DIRS = [p for p in Path(".").iterdir() if p.is_dir() and p.name.startswith("llm_drafts")]
all_data_list = []

for folder in MODEL_DIRS:
    model_tag = folder.name.replace("llm_drafts_", "").upper()
    for csv_path in folder.glob("*.csv"):
        v_key = "general" if any(x in csv_path.name.lower() for x in ["expert", "general"]) \
                else next((v for v in PLOT_ORDER if v in csv_path.name.lower()), None)
        if not v_key: continue
        ldf = pd.read_csv(csv_path)
        ldf['essay_id'] = ldf['essay_id'].apply(super_clean_id)
        ai_col = 'output_text' if 'output_text' in ldf.columns else 'final_llm_text'
        merged = pd.merge(df_human, ldf[['essay_id', ai_col]], on='essay_id').rename(columns={ai_col: 'ai_text'})
        merged['llm_model'], merged['mode'] = model_tag, v_key
        merged['feedback'] = "WITH" if "with_FB" in csv_path.name else "WITHOUT"
        all_data_list.append(merged)

df_master = pd.concat(all_data_list)


In [42]:
import pandas as pd

# Load the dataset
df = pd.read_csv('dataframe_essays_llm_model.csv')

# Defining 10 diverse examples focusing on General/Minimal modes across multiple models
target_examples = [
    {"id": 1, "model": "GEMINI-2.5-FLASH", "mode": "minimal", "issue": "Devalued Opposition: Changed 'valid points' to 'merit', subtly weakening the author's neutral stance."},
    {"id": 15, "model": "GEMINI-2.5-FLASH", "mode": "general", "issue": "Bias Injection: Reframed a neutral list as a 'complex dilemma' with 'considerable risks'."},
    {"id": 50, "model": "GPT-5-MINI", "mode": "general", "issue": "Fact Hallucination: Added historical facts (Wright brothers) changing the essay from philosophical to historical survey."},
    {"id": 33, "model": "GEMINI-2.5-FLASH", "mode": "general", "issue": "Tone Sanitization: Replaced the 'Sci-Fi' wonder metaphor with a dry analysis of 'technological progress'."},
    {"id": 4, "model": "GPT-5-MINI", "mode": "general", "issue": "Argument Expansion: Injected safety claims about 'human error' statistics not present in the human draft."},
    {"id": 22, "model": "GPT-5-MINI", "mode": "minimal", "issue": "Stance Hardening: Rephrased a tentative 'might deter' into a definitive statement about 'significant drawbacks'."},
    {"id": 41, "model": "GEMINI-2.5-FLASH", "mode": "general", "issue": "Value Shift: Changed the 'privilege' of independence into a 'burden' of 'considerable responsibility and danger'."},
    {"id": 80, "model": "GPT-5-MINI", "mode": "general", "issue": "Voice Erasure: Replaced an emotional rhetorical question with a clinical analysis of risk mitigation."},
    {"id": 1, "model": "GPT-5-MINI", "mode": "general", "issue": "Certainty Injection: Changed a debate about 'infancy' into a factual claim about 'accelerated development'."},
    {"id": 33, "model": "GPT-5-MINI", "mode": "minimal", "issue": "Nuance Polishing: Removed the human's emphasis on 'appreciating most' to make the claim sound more clinical."},
    {"id": 29, "model": "GPT-5-MINI", "mode": "minimal", "issue": "Nuance Polishing: Removed the human's emphasis on 'appreciating most' to make the claim sound more clinical."}

]

print("--- 10 EXAMPLES OF DRASTIC CLAIM CHANGES (GEMINI & GPT-5-MINI) ---\n")

for ex in target_examples:
    match = df[(df['essay_id'] == ex['id']) & 
               (df['llm_model'] == ex['model']) & 
               (df['mode'] == ex['mode'])]
    
    if not match.empty:
        row = match.iloc[0]
        print(f"ID: {ex['id']} | MODEL: {ex['model']} | MODE: {ex['mode'].upper()}")
        print(f"THE ISSUE: {ex['issue']}")
        
        # Snippets for comparison
        h_text = str(row['draft1']).replace('\n', ' ')
        a_text = str(row['ai_text']).replace('\n', ' ')
        
        print(f"  HUMAN: \"{h_text}...\"")
        print(f"  AI:    \"{a_text}...\"")
        print("-" * 80 + "\n")

--- 10 EXAMPLES OF DRASTIC CLAIM CHANGES (GEMINI & GPT-5-MINI) ---

ID: 1 | MODEL: GEMINI-2.5-FLASH | MODE: MINIMAL
THE ISSUE: Devalued Opposition: Changed 'valid points' to 'merit', subtly weakening the author's neutral stance.
  HUMAN: "The topic of self-driving cars is highly debated especially in light of recent event with the death of an Arizona woman at the hands of a self-driving Uber. Those for self-driving cars argue that they reduce the rate of accidents as a result of distracted drivers, improve traffic flow, are cost saving in terms of time and monetary value from insurance, and could allow greater access to transportation for the disabled and elderly. Those against self-driving cars argue the system does not eliminate accidents, that there is a learning curve with the technology, that it would take away profit from other sectors of the economy, and that there are instances in which the technology is not equipped to make a decision. While both sides make valid points, the b